In [18]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [19]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
JINA_API_KEY=os.getenv("GROQ_API_KEY")


In [20]:
DATA_FILE_PATH=os.path.join("data","hr_policy.txt")

In [21]:
loader=TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents=loader.load()
print("DATA LOADED")
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

In [5]:
print(len(documents))
print(documents[0].page_content)

1
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of

In [6]:
print(documents[0].metadata)
print(f"Totalcharacters in document: {len(documents[0].page_content)}")

{'source': 'data\\hr_policy.txt'}
Totalcharacters in document: 2597


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=text_splitter.split_documents(documents)

In [8]:
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'data\\hr_policy.txt'}


NOW EACH SPLITTED CHUNK IS A DOCUMENT THA HAS PAGE CONTENT AND METADATA

EMBED THE DATA


In [23]:
from langchain_community.embeddings import JinaEmbeddings
embedding_model=JinaEmbeddings(model_name="jina-embeddings-v4")
print("Give embedding model",embedding_model.model_name)

Give embedding model jina-embeddings-v4


In [24]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(chunks,embedding_model)
print("CHUNKS ARE STORED",vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [25]:
text_query="How many sick leaves employee get"

top_matches=vector_store.similarity_search(text_query,k=2)
for i , match in enumerate(top_matches,start=1):
    print(f"--Match{i}---")
    print(match.page_content)
    print()

--Match1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--Match2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [26]:
retriever=vector_store.as_retriever(search_kwargs={"K":3})
def search_hr_policy(question:str)->str:
    """
    Search the HR Policy document forinformationabout leave,work from home,probation,notice period,reimbursement,code of conduct,holidays, or exit process
    """
    matching_chunks=retriever.invoke(question)
    return"\n\n".join(chunk.page_content for chunk in matching_chunks)

In [27]:
#DATA RETRIVAL
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-120b",
             temperature=0)
llm.model_name

'openai/gpt-oss-120b'

In [28]:
from langchain.agents import create_agent
hr_Assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=
    """
    You are a friendly hr assistant.Always use the search_hr_policy tool to look up the facts before answering.If the answer isnt in the search results, say you dont know instead of guessing

    """
)

In [29]:
response=hr_Assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"tell me about leave policy how to apply for leave"
            }

        ]
    }
)

In [30]:
print(response["messages"][-1].content)

**Leave Policy Overview**

| Type of leave | Entitlement | Key points |
|---------------|------------|------------|
| **Annual (paid) leave** | 20 days per calendar year | • Unused days can be carried forward, but only up to **5 days** into the next year.<br>• Must be requested **at least 5 working days** before the intended start date. |
| **Sick leave** | 10 paid days per year | • Separate from annual leave.<br>• If you are sick for **more than 2 consecutive days**, you must provide a medical certificate. |

**How to Apply for Leave**

1. **Log into the HR portal** – the company’s dedicated HR self‑service system is the only approved channel for leave requests.  
2. **Select “Leave Request”** – choose the appropriate leave type (Annual or Sick).  
3. **Enter the dates** – specify the start and end dates of the leave.  
4. **Provide any required documentation** – for sick leave longer than 2 days, attach a medical certificate.  
5. **Submit the request** – the system will route the re